## EDA

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Datasets" / 'GEFCOM2017'
import pandas as pd
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt

In [ ]:
#path = "C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/"
gef  = pd.read_csv(f"Gef2017.csv")#path}

# Example data
df = gef.copy()

# Define mapping rules
level_2 = ['CT', 'MASS', 'ME', 'NH', 'RI', 'VT']
level_1 = ['TOTAL']

# Use numpy.select or df.apply; the vectorized form is faster here:
df['level'] = (
    df['zone'].apply(
        lambda x: 2 if x in level_2 else (1 if x in level_1 else 3)
    )
)

df

In [ ]:
df['ts'] = pd.to_datetime(df['ts'].astype(str).str.strip(), format='mixed')
df = df.sort_values(['zone', 'ts']).copy()
df['demand_smooth'] = df.groupby('zone')['demand'].transform(
    lambda values: values.rolling(window=24*7, min_periods=1).mean()
)

# Plot only three levels, selected as needed
levels = sorted(df['level'].unique())#[:3]
zones = sorted(df['zone'].unique())

# Use a fixed color mapping for consistent colors across subplots
cmap = plt.cm.tab20 if len(zones) > 10 else plt.cm.tab10
colors = cmap(np.linspace(0, 1, len(zones)))
zone2color = {z: colors[i % len(colors)] for i, z in enumerate(zones)}

# Three rows and one column
ncols, nrows = 1, 3
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(17, 9), sharex=True, sharey=False)
if nrows == 1:
    axes = [axes]  # Ensure the axes are iterable

# Collect legend handles
handle_map = {}

for i, level in enumerate(levels):
    ax = axes[i]
    sub = df[df['level'] == level]

    for z in zones:
        zone_data = sub[sub['zone'] == z]
        if not zone_data.empty:
            line, = ax.plot(zone_data['ts'], zone_data['demand_smooth'],
                            color=zone2color[z], linewidth=1.3, label=z, alpha=0.95)
            # Record each zone handle only on its first occurrence
            if z not in handle_map:
                handle_map[z] = line

    ax.set_title(f"Level {level}", fontsize=16, weight='bold')
    #ax.set_xlabel("Time")
    if level == 2:
        ax.set_ylabel("Demand (MW)", fontsize=14, weight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3)

    # Black borders around subplots
    for spine in ax.spines.values():
        spine.set_edgecolor('grey')
        spine.set_linewidth(1.2)

# Remove unused subplots if fewer than 3 levels are available
for j in range(len(levels), nrows):
    fig.delaxes(axes[j])

# Overall title
#fig.suptitle("Hourly Electricity Demand by Level (One Week Rolling Mean)", fontsize=16, weight='bold', y=0.98)
#fig.ylabel("Demand (MW)", fontsize=13, weight='bold')
# One legend on the right containing all zones
handles = list(handle_map.values())
labels = list(handle_map.keys())
#fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.75), frameon=False)#title="Zone"

leg = fig.legend(
    handles, labels,
    #title="Zone",
    loc='center left',
    bbox_to_anchor=(1, 0.8),
    frameon=True,
    fontsize=13#,
    #title_fontsize=11
)

# Add a black border to the legend
frame = leg.get_frame()
frame.set_edgecolor('grey')
frame.set_linewidth(1.2)
frame.set_alpha(1)
frame.set_facecolor('white')

# Leave space for the legend on the right

plt.tight_layout()#rect=[0, 0, 0.85, 0.95]
plt.savefig(str(DATA_DIR / "Hourly_Electricity_Demand.pdf"), format="pdf", bbox_inches="tight")
plt.show()

## Preprocessing - deduplicated hierarchy


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from hierarchicalforecast.utils import aggregate

BASE_PATH = PROJECT_ROOT
DATA_FILE = BASE_PATH / "Datasets/GEFCOM2017/Gef2017.csv"
if not DATA_FILE.exists():
    DATA_FILE = Path(str(DATA_DIR / "Gef2017.csv"))

gef2 = pd.read_csv(DATA_FILE)
gef = gef2.iloc[:, :-10].copy()
gef = gef.rename(columns={"ts": "ds", "demand": "y"})
gef["region"] = gef["zone"]
keep_regions = ["CT", "ME", "NEMASSBOST", "NH", "RI", "SEMASS", "VT", "WCMASS"]
gef = gef.loc[gef["zone"].isin(keep_regions), :].copy()
gef.loc[gef["zone"].str.contains("MASS", na=False), "zone"] = "MASS"
gef["total"] = "Total"
gef = gef[["ds", "zone", "y", "drybulb", "dewpnt", "region", "total"]]

SPEC = [
    ["total"],                 # Level 1
    ["total", "zone"],         # Level 2
    ["total", "zone", "region"] # Level 3
]
Y_raw, S_raw, tags_raw = aggregate(gef, SPEC)

duplicate_leaf_to_parent = {
    "Total/CT/CT": "Total/CT",
    "Total/ME/ME": "Total/ME",
    "Total/NH/NH": "Total/NH",
    "Total/RI/RI": "Total/RI",
    "Total/VT/VT": "Total/VT",
}
drop_ids = list(duplicate_leaf_to_parent.keys())

Y_df = Y_raw.loc[~Y_raw["unique_id"].isin(drop_ids), ["unique_id", "ds", "y"]].copy()
Y_df["ds"] = pd.to_datetime(Y_df["ds"], format="mixed")
Y_df["levels"] = Y_df["unique_id"].str.count("/") + 1

bottom_ids = [duplicate_leaf_to_parent.get(c, c) for c in S_raw.columns if c != "unique_id"]

S_df = S_raw.rename(columns=duplicate_leaf_to_parent).copy()
S_df["unique_id"] = S_df["unique_id"].replace(duplicate_leaf_to_parent)
S_df = S_df.drop_duplicates(subset="unique_id", keep="first").copy()

node_order = [uid for uid in S_raw["unique_id"].tolist() if uid not in drop_ids]
S_df = (
    S_df.set_index("unique_id")
        .loc[node_order, bottom_ids]
        .reset_index()
)

tags = {
    "total": np.array(["Total"], dtype=object),
    "total/zone": np.array([
        "Total/CT", "Total/MASS", "Total/ME", "Total/NH", "Total/RI", "Total/VT"
    ], dtype=object),
    "bottom": np.array(bottom_ids, dtype=object),
}

label2id = {uid: i for i, uid in enumerate(S_df["unique_id"].tolist())}
Y_df["partition_id"] = Y_df["unique_id"].map(label2id).astype(int)
Y_df = Y_df.sort_values(["partition_id", "ds"]).reset_index(drop=True)

# =========================
# Sanity checks
# =========================
assert Y_df["unique_id"].nunique() == S_df.shape[0], "Y_df and S_df have different node counts"
assert set(bottom_ids).issubset(set(Y_df["unique_id"])), "Some bottom_ids are missing from Y_df"
assert not Y_df.duplicated(["unique_id", "ds"]).any(), "Y_df contains duplicate (unique_id, ds) pairs"
assert not S_df["unique_id"].duplicated().any(), "S_df contains duplicate unique_id values"

# Check that S @ bottom reconstructs all aggregate nodes.
wide = Y_df.pivot(index="ds", columns="unique_id", values="y")
S_mat = S_df.set_index("unique_id")[bottom_ids].astype(float)
reconciled = wide[bottom_ids].to_numpy() @ S_mat.T.to_numpy()
target = wide[S_df["unique_id"].tolist()].to_numpy()
max_agg_error = np.nanmax(np.abs(reconciled - target))
print(f"Deduplicated hierarchy: {S_df.shape[0]} nodes, {len(bottom_ids)} bottom series")
print(f"max aggregation error = {max_agg_error:.3e}")
assert max_agg_error < 1e-6, "The deduplicated S matrix cannot reconstruct the aggregate levels"

Y_df


In [ ]:
Y_df.to_csv(str(DATA_DIR / "GEF2017_univarivate.csv"), index=False)


In [ ]:
S_df.to_csv(str(DATA_DIR / "GEF_S.csv"), index=False)


In [ ]:
S_df

In [ ]:
import pickle
with open(str(DATA_DIR / "tags_dedup.pkl"), "wb") as f:
    pickle.dump(tags, f)


In [ ]:
def Datacollector(dff, lags: int = 48):
    """Build lag features by partition_id without hard-coding the node count."""
    out = dff.sort_values(["partition_id", "ds"]).copy()
    for lag in range(1, lags + 1):
        out[f"lags_{lag}"] = out.groupby("partition_id")["y"].shift(lag)
    return out

# There are 10 nodes after deduplication, so range(15) cannot be used here.
dff = Datacollector(Y_df, lags=48).dropna()
dff = dff[dff["ds"] >= pd.Timestamp("2017-01-01")].copy()

dff.to_csv(str(DATA_DIR / "GEF2017_170101_dedup.csv"), index=False)
dff


In [ ]:
dff